In [1]:
#prep relevent packages 
from pathlib import Path
import pandas as pd
import re

In [2]:
# package to add to kernel to read pdfs. 
# Can also do %conda install pdfplumber if running on windows 
try:
    import pdfplumber
except Importerror:
    %pip install pdfplumber
    import pdfplumber 

In [3]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw' 
processed_fldr = root_fldr / 'data' / 'processed'

# import cfp pdfs data
general_plan = raw_fldr / 'cfp_data' / 'cfp_5_years_plan_data.pdf'


In [6]:
table_settings = {
    'vertical_strategy': 'text',
    'horizontal_strategy': 'text',
}
rows = []  # reset each run so reruns don't duplicate rows

#help provided with some AI, particularly with zip_cell check to get unified, scraped dataframe
with pdfplumber.open(general_plan) as pdf:
    for page in pdf.pages:
        for table in page.extract_tables(table_settings):
            for row in table:
                zip_cell = (row[0] or '').strip()
                if re.fullmatch(r'\d{5}', zip_cell):
                    # clean each cell: None -> '', drop spaces, commas, and %
                    clean_row = [(cell or '').replace(' ', '').replace(',', '').replace('%', '')
                                 for cell in row]
                    rows.append(clean_row)

cols = ['zip',
        'growth_2025', 'policies_2025',
        'growth_2024', 'policies_2024',
        'growth_2023', 'policies_2023',
        'growth_2022', 'policies_2022',
        'policies_2021']
general_plan = pd.DataFrame(rows, columns=cols)


In [7]:
#check dataframe to see if it can be read  
general_plan.head()

,zip,growth_2025,policies_2025,growth_2024,policies_2024,growth_2023,policies_2023,growth_2022,policies_2022,policies_2021
0,94501,112,242,39,114,8,82,-7,76,82
1,94502,71,12,250,7,0,2,100,2,1
2,94536,84,116,200,63,75,21,9,12,11
3,94538,55,45,142,29,100,12,20,6,5
4,94539,90,116,221,61,111,19,200,9,3


In [8]:
#converting df into appropriate variable types 

general_plan_clean = (
    general_plan
    .apply(pd.to_numeric, errors='coerce') #converting strings to numbers, '-' becomes NaN
    .assign( #converting into percentages 
        growth_2025 = lambda d: d['growth_2025'] / 100,
        growth_2024 = lambda d: d['growth_2024'] / 100,
        growth_2023 = lambda d: d['growth_2023'] / 100,
        growth_2022 = lambda d: d['growth_2022'] / 100 
    )
    .astype({
        'growth_2025': 'float64',
        'policies_2025': 'Int64',
        'growth_2024': 'float64',
        'policies_2024': 'Int64',
        'growth_2023': 'float64',
        'policies_2023': 'Int64',
        'growth_2022': 'float64',
        'policies_2022': 'Int64',
        'policies_2021': 'Int64'
        })
)

In [9]:
general_plan_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   zip            1682 non-null   int64  
 1   growth_2025    1682 non-null   float64
 2   policies_2025  1667 non-null   Int64  
 3   growth_2024    1682 non-null   float64
 4   policies_2024  1656 non-null   Int64  
 5   growth_2023    1682 non-null   float64
 6   policies_2023  1607 non-null   Int64  
 7   growth_2022    1682 non-null   float64
 8   policies_2022  1577 non-null   Int64  
 9   policies_2021  1551 non-null   Int64  
dtypes: Int64(5), float64(4), int64(1)
memory usage: 139.7 KB


In [10]:
general_plan_clean.head()

,zip,growth_2025,policies_2025,growth_2024,policies_2024,growth_2023,policies_2023,growth_2022,policies_2022,policies_2021
0,94501,1.12,242,0.39,114,0.08,82,-0.07,76,82
1,94502,0.71,12,2.50,7,0.00,2,1.00,2,1
2,94536,0.84,116,2.00,63,0.75,21,0.09,12,11
3,94538,0.55,45,1.42,29,1.00,12,0.20,6,5
4,94539,0.90,116,2.21,61,1.11,19,2.00,9,3


In [11]:
#check to see if numbers from pull are aligned with pdf 
# Plans from pdf
# 2025 plans = 642,010
# 2024 plans = 463,159 
# 2023 plans = 330,265
total_2025_policies = general_plan_clean['policies_2025'].sum()
total_2024_policies = general_plan_clean['policies_2024'].sum()
total_2023_policies = general_plan_clean['policies_2023'].sum()

print('2025 plans:',total_2025_policies)
print('2024 plans:',total_2024_policies)
print('2023 plans:',total_2023_policies)



2025 plans: 642010
2024 plans: 463159
2023 plans: 330265


In [ ]:
#converting to processed folder csv
general_plan_clean.to_csv(processed_fldr / 'cfp_general_plans_cleaned.csv')
